# WSI-Level Inference & Spatial Visualization

학습된 Two-Head 모델을 WSI에 적용하여 marker gene의 spatial expression을 예측하고,  
Xenium 공간전사체 데이터(GT)와 비교하여 시각화합니다.

In [ ]:
# ===== 1. 설정 및 라이브러리 임포트 =====
import os
import numpy as np
import pandas as pd
import anndata as ad
from glob import glob
from PIL import Image
from tqdm import tqdm
from scipy import stats

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.transforms import ToTensor
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

Image.MAX_IMAGE_PIXELS = None  # 대용량 이미지 허용

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# ===== 2. 경로 및 하이퍼파라미터 설정 =====
# H&E 슬라이드 경로
slide_path = '../../data/portrai/POR-2025-SER-1041/2. Xenium processed data/P-B0-R-AI-B01-54-X_C00/P-B0-R-AI-B01-54-X_C00.tiff'

# Xenium processed data 경로 (h5ad)
xenium_processed_dir = '../../data/portrai/POR-2025-SER-1041/2. Xenium processed data'
# 해당 슬라이드의 Xenium capture areas
xenium_slide_prefix = 'P-B0-R-AI-B01-54-X'

# 모델 체크포인트 경로
model_path = '../../model/marker_gene_spatial_expression/'

NUM_GENES = 15
ENCODER_NAME = 'tu-convnext_tiny'
BATCH_SIZE = 32
PATCH_SIZE = 512       # 20x patch size (pixels)
TARGET_MPP = 0.5       # 20x magnification ≈ 0.5 µm/pixel
CONTEXT_SCALE = 4      # 5x context = 4x wider than 20x

MARKER_GENES = [
    'EPCAM', 'KRT8', 'KRT18', 'KRT19',
    'COL1A1', 'COL1A2', 'FAP', 'ACTA2',
    'CD3D', 'CD3E', 'CD8A',
    'MS4A1', 'CD79A',
    'CD68',
    'PECAM1'
]

GENE_GROUPS = {
    'Epithelial/Tumor': [0, 1, 2, 3],
    'Stromal/Fibroblast': [4, 5, 6, 7],
    'T Cell': [8, 9, 10],
    'B Cell': [11, 12],
    'Macrophage': [13],
    'Endothelial': [14],
}

In [ ]:
# ===== 3. 이미지 로드 및 정보 확인 =====
print('Loading image into memory...')
slide_img = np.array(Image.open(slide_path))  # (H, W, 3) uint8
H_img, W_img = slide_img.shape[:2]
print(f'Image shape: {slide_img.shape} (H={H_img}, W={W_img})')
print(f'dtype: {slide_img.dtype}, memory: {slide_img.nbytes / 1e9:.1f} GB')

# Xenium morphology pixel size
slide_mpp = 0.2125  # µm/pixel (Xenium)
print(f'Slide MPP: {slide_mpp:.4f} µm/pixel')

# 20x에 해당하는 스케일 계산
downsample_20x = TARGET_MPP / slide_mpp
print(f'20x downsample factor: {downsample_20x:.2f}')

# level 0 기준 patch size (20x 해상도에 맞게)
patch_size_l0 = int(PATCH_SIZE * downsample_20x)
context_size_l0 = patch_size_l0 * CONTEXT_SCALE
print(f'Patch size at level 0: {patch_size_l0} pixels (20x: {PATCH_SIZE})')
print(f'Context size at level 0: {context_size_l0} pixels (5x: {PATCH_SIZE})')

# 썸네일 표시
thumb_scale = 1500 / max(W_img, H_img)
thumb = Image.fromarray(slide_img).resize(
    (int(W_img * thumb_scale), int(H_img * thumb_scale)), Image.LANCZOS)
plt.figure(figsize=(10, 6))
plt.imshow(thumb)
plt.title(f'Image Thumbnail ({W_img}x{H_img})')
plt.axis('off')
plt.show()

In [ ]:
# ===== 4. 모델 정의 및 체크포인트 로드 =====
class MultiScaleRegressionModel(nn.Module):
    def __init__(self, encoder_name='tu-convnext_tiny', num_genes=15):
        super().__init__()
        self.encoder_20x = smp.encoders.get_encoder(encoder_name, in_channels=3, weights='imagenet')
        self.encoder_5x = smp.encoders.get_encoder(encoder_name, in_channels=3, weights='imagenet')

        enc_channels = self.encoder_20x.out_channels
        feat_dim = enc_channels[-1]

        self.head_prop = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, num_genes),
        )
        self.head_int = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, num_genes),
            nn.Sigmoid()
        )

    def forward(self, x_20x, x_5x):
        feat_20x = self.encoder_20x(x_20x)
        feat_5x = self.encoder_5x(x_5x)
        fused = feat_20x[-1] + feat_5x[-1]
        pooled = F.adaptive_avg_pool2d(fused, 1).flatten(1)
        prop_logits = self.head_prop(pooled)
        intensity = self.head_int(pooled)
        return prop_logits, intensity


model = MultiScaleRegressionModel(ENCODER_NAME, NUM_GENES).to(device)

ckpt_path = os.path.join(model_path, 'train_2head_best.pt')
assert os.path.exists(ckpt_path), f'Checkpoint not found: {ckpt_path}'
state_dict = torch.load(ckpt_path, map_location=device)
model.load_state_dict(state_dict, strict=False)
model.eval()
print(f'Checkpoint loaded: {ckpt_path}')
print(f'Total params: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ===== 5. 그리드 패치 좌표 생성 및 조직 영역 필터링 =====
W, H = W_img, H_img

# 그리드 좌표 생성 (level 0 기준)
patch_coords = []  # (x, y) at level 0
for y in range(0, H - patch_size_l0 + 1, patch_size_l0):
    for x in range(0, W - patch_size_l0 + 1, patch_size_l0):
        patch_coords.append((x, y))
print(f'Total grid patches: {len(patch_coords)}')

# 조직 영역 필터링 (썸네일 기반)
thumb_size = 1024
thumb_scale_x = W / thumb_size
thumb_scale_y = H / thumb_size
thumb_np = np.array(Image.fromarray(slide_img).resize(
    (thumb_size, int(H / thumb_scale_x)), Image.LANCZOS))
thumb_gray = np.mean(thumb_np[:, :, :3], axis=2)
tissue_mask = (thumb_gray < 220) & (thumb_gray > 30)

scale_x = W / thumb_np.shape[1]
scale_y = H / thumb_np.shape[0]

tissue_coords = []
for (x, y) in patch_coords:
    tx = int((x + patch_size_l0 / 2) / scale_x)
    ty = int((y + patch_size_l0 / 2) / scale_y)
    tx = min(tx, thumb_np.shape[1] - 1)
    ty = min(ty, thumb_np.shape[0] - 1)
    if tissue_mask[ty, tx]:
        tissue_coords.append((x, y))

print(f'Tissue patches (after filtering): {len(tissue_coords)}')

# 조직 마스크 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(thumb_np)
axes[0].set_title('Thumbnail')
axes[0].axis('off')
axes[1].imshow(tissue_mask, cmap='gray')
axes[1].set_title(f'Tissue Mask ({len(tissue_coords)} patches)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ===== 6. 패치 추출 및 모델 추론 (numpy 슬라이싱) =====
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast

to_tensor = ToTensor()

def extract_patch_np(img, x, y, size, target_size=512):
    """numpy 배열에서 패치 추출 후 target_size로 리사이즈"""
    patch = img[y:y+size, x:x+size]  # (H, W, 3)
    if patch.shape[0] != target_size or patch.shape[1] != target_size:
        patch = np.array(Image.fromarray(patch).resize(
            (target_size, target_size), Image.LANCZOS))
    return patch


class ImagePatchDataset(Dataset):
    def __init__(self, img, coords, patch_size_l0, context_size_l0,
                 W, H, patch_size=512):
        self.img = img
        self.coords = coords
        self.patch_size_l0 = patch_size_l0
        self.context_size_l0 = context_size_l0
        self.W, self.H = W, H
        self.patch_size = patch_size

    def __len__(self):
        return len(self.coords)

    def __getitem__(self, idx):
        x, y = self.coords[idx]

        # 20x patch
        patch_20x = extract_patch_np(self.img, x, y, self.patch_size_l0, self.patch_size)

        # 5x context patch (동일 중심, 4배 넓은 영역)
        cx = x + self.patch_size_l0 // 2
        cy = y + self.patch_size_l0 // 2
        ctx_x = max(0, cx - self.context_size_l0 // 2)
        ctx_y = max(0, cy - self.context_size_l0 // 2)
        ctx_x = min(ctx_x, self.W - self.context_size_l0)
        ctx_y = min(ctx_y, self.H - self.context_size_l0)
        ctx_x = max(0, ctx_x)
        ctx_y = max(0, ctx_y)
        patch_5x = extract_patch_np(self.img, ctx_x, ctx_y, self.context_size_l0, self.patch_size)

        img_20x = to_tensor(patch_20x)
        img_5x = to_tensor(patch_5x)
        return img_20x, img_5x


dataset = ImagePatchDataset(slide_img, tissue_coords, patch_size_l0, context_size_l0,
                            W, H, PATCH_SIZE)

loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=4, pin_memory=True, prefetch_factor=2)

# AMP 추론
all_pred_prop = []
all_pred_int = []

with torch.no_grad():
    for imgs_20x, imgs_5x in tqdm(loader, desc='Inference'):
        imgs_20x = imgs_20x.to(device, non_blocking=True)
        imgs_5x = imgs_5x.to(device, non_blocking=True)

        with autocast(device_type='cuda'):
            pred_prop_logits, pred_int = model(imgs_20x, imgs_5x)

        pred_prop = torch.sigmoid(pred_prop_logits)
        all_pred_prop.append(pred_prop.cpu())
        all_pred_int.append(pred_int.cpu())

all_pred_prop = torch.cat(all_pred_prop).float().numpy()
all_pred_int = torch.cat(all_pred_int).float().numpy()
all_coords = np.array(tissue_coords)

print(f'Inference complete: {len(all_coords)} patches')
print(f'Pred proportion shape: {all_pred_prop.shape}')
print(f'Pred intensity shape: {all_pred_int.shape}')

In [ ]:
# ===== 7. Xenium GT 데이터 로드 (h5ad, P-B0-R-AI-B01-54-X_C00) =====
import scipy.sparse

h5ad_path = os.path.join(xenium_processed_dir, f'{xenium_slide_prefix}_C00',
                         f'{xenium_slide_prefix}_C00_labeled.h5ad')
print(f'Loading: {h5ad_path}')
adata_all = ad.read_h5ad(h5ad_path)
print(f'Cells: {adata_all.n_obs}, Genes: {adata_all.n_vars}')

# 세포 좌표 (µm 단위)
cell_x_um = adata_all.obs['x_centroid'].values.astype(np.float64)
cell_y_um = adata_all.obs['y_centroid'].values.astype(np.float64)
print(f'Cell coordinate range (µm): x=[{cell_x_um.min():.1f}, {cell_x_um.max():.1f}], '
      f'y=[{cell_y_um.min():.1f}, {cell_y_um.max():.1f}]')

# 발현 행렬 (cells x genes) → CSC로 변환 (gene 슬라이싱 효율)
gene_names = list(adata_all.var.index)
matrix = scipy.sparse.csc_matrix(adata_all.X)  # (cells x genes), CSC
print(f'Expression matrix: {matrix.shape} (cells x genes)')

# 마커 유전자 인덱스 매핑
marker_gene_idx = {}
available_markers = []
missing_markers = []
for gi, gene in enumerate(MARKER_GENES):
    if gene in gene_names:
        marker_gene_idx[gene] = gene_names.index(gene)
        available_markers.append((gi, gene))
    else:
        missing_markers.append(gene)

print(f'\nAvailable marker genes ({len(available_markers)}/{NUM_GENES}):')
for gi, gene in available_markers:
    print(f'  [{gi:2d}] {gene}')
if missing_markers:
    print(f'Missing marker genes: {missing_markers}')

In [ ]:
# ===== 8. Xenium 좌표를 이미지 픽셀 좌표로 변환 =====
# Xenium 좌표(µm) → 이미지 pixel = xenium_coord(µm) / slide_mpp
# slide_mpp = 0.2125 µm/pixel (Xenium morphology image)

print(f'Image physical size: {W * slide_mpp:.0f} x {H * slide_mpp:.0f} µm')

cell_x_pixel = cell_x_um / slide_mpp
cell_y_pixel = cell_y_um / slide_mpp

print(f'Cell pixel range: x=[{cell_x_pixel.min():.0f}, {cell_x_pixel.max():.0f}], '
      f'y=[{cell_y_pixel.min():.0f}, {cell_y_pixel.max():.0f}]')
print(f'Image pixel range: x=[0, {W}], y=[0, {H}]')

# 이미지 범위 내 세포 비율 확인
in_bounds = ((cell_x_pixel >= 0) & (cell_x_pixel < W) &
             (cell_y_pixel >= 0) & (cell_y_pixel < H))
print(f'Cells within image bounds: {in_bounds.sum()} / {len(cell_x_pixel)} '
      f'({in_bounds.sum()/len(cell_x_pixel)*100:.1f}%)')

In [ ]:
# ===== 9. 패치별 GT 라벨 계산 (Xenium 기반) =====
# 각 패치 영역 내 세포들의 발현을 집계하여 proportion, intensity 계산
# 주의: matrix shape = (cells x genes), CSC format

gt_prop = np.zeros((len(all_coords), NUM_GENES), dtype=np.float32)
gt_int = np.zeros((len(all_coords), NUM_GENES), dtype=np.float32)
cells_per_patch = np.zeros(len(all_coords), dtype=np.int32)

# 전체 세포별 total counts 미리 계산 (CPM 정규화용)
total_counts_all = np.array(matrix.sum(axis=1)).flatten()  # (n_cells,)

for pi, (px, py) in enumerate(tqdm(all_coords, desc='Computing GT labels')):
    # 패치 영역 내 세포 찾기
    x_min, x_max = px, px + patch_size_l0
    y_min, y_max = py, py + patch_size_l0

    mask = ((cell_x_pixel >= x_min) & (cell_x_pixel < x_max) &
            (cell_y_pixel >= y_min) & (cell_y_pixel < y_max))
    cell_indices = np.where(mask)[0]
    n_cells = len(cell_indices)
    cells_per_patch[pi] = n_cells

    if n_cells == 0:
        continue

    # 해당 세포들의 total counts
    cell_totals = total_counts_all[cell_indices]

    # 해당 세포들의 발현 데이터 추출
    for gi, gene in enumerate(MARKER_GENES):
        if gene not in marker_gene_idx:
            continue
        feat_idx = marker_gene_idx[gene]
        # matrix: (cells x genes), CSC → gene 열 슬라이싱 효율적
        expr_vals = matrix[cell_indices, feat_idx].toarray().flatten()

        # Head A: Proportion (발현 양성 비율)
        n_positive = np.sum(expr_vals > 0)
        gt_prop[pi, gi] = n_positive / n_cells

        # Head B: Intensity (양성 세포의 평균 발현 강도)
        if n_positive > 0:
            positive_mask = expr_vals > 0
            positive_vals = expr_vals[positive_mask]
            positive_total = cell_totals[positive_mask]
            cp10k = positive_vals / (positive_total + 1e-8) * 10000
            intensity = np.clip(np.mean(np.log1p(cp10k)), 0, 10) / 10.0
            gt_int[pi, gi] = intensity

# 유효 패치 (1개 이상 세포 포함)
valid_mask = cells_per_patch > 0
print(f'Patches with cells: {valid_mask.sum()} / {len(all_coords)}')
print(f'Cells per patch (valid): mean={cells_per_patch[valid_mask].mean():.1f}, '
      f'median={np.median(cells_per_patch[valid_mask]):.0f}')

In [ ]:
# ===== 10. 유전자별 PCC & MAE 테이블 (유효 패치 기준) =====
pred_prop_valid = all_pred_prop[valid_mask]
pred_int_valid = all_pred_int[valid_mask]
gt_prop_valid = gt_prop[valid_mask]
gt_int_valid = gt_int[valid_mask]

pcc_prop_list = []
pcc_int_list = []
mae_prop_list = []
mae_int_list = []

print(f'\n{"Gene":>10s} | {"PCC(prop)":>10s} {"MAE(prop)":>10s} | {"PCC(int)":>10s} {"MAE(int)":>10s} | {"Available":>10s}')
print('-' * 75)
for gi, gene in enumerate(MARKER_GENES):
    mae_p = np.mean(np.abs(pred_prop_valid[:, gi] - gt_prop_valid[:, gi]))
    mae_i = np.mean(np.abs(pred_int_valid[:, gi] - gt_int_valid[:, gi]))
    mae_prop_list.append(mae_p)
    mae_int_list.append(mae_i)

    if gene in marker_gene_idx and gt_prop_valid[:, gi].std() > 1e-6:
        r_p, _ = stats.pearsonr(pred_prop_valid[:, gi], gt_prop_valid[:, gi])
        r_i, _ = stats.pearsonr(pred_int_valid[:, gi], gt_int_valid[:, gi])
        avail = 'Yes'
    else:
        r_p, r_i = float('nan'), float('nan')
        avail = 'No' if gene not in marker_gene_idx else 'Low var'
    pcc_prop_list.append(r_p)
    pcc_int_list.append(r_i)

    print(f'{gene:>10s} | {r_p:10.4f} {mae_p:10.4f} | {r_i:10.4f} {mae_i:10.4f} | {avail:>10s}')

# 유효 유전자만 평균
valid_genes = [i for i in range(NUM_GENES) if not np.isnan(pcc_prop_list[i])]
print('-' * 75)
print(f'{"Mean":>10s} | {np.nanmean(pcc_prop_list):10.4f} {np.nanmean(mae_prop_list):10.4f} | '
      f'{np.nanmean(pcc_int_list):10.4f} {np.nanmean(mae_int_list):10.4f} | '
      f'{len(valid_genes):>10d}')

In [ ]:
# ===== 11. Spatial Heatmap: 예측 Proportion =====
# 패치 중심 좌표를 그리드 인덱스로 변환
center_x = (all_coords[:, 0] + patch_size_l0 / 2) / patch_size_l0
center_y = (all_coords[:, 1] + patch_size_l0 / 2) / patch_size_l0

n_cols = int(W / patch_size_l0)
n_rows = int(H / patch_size_l0)

def make_spatial_map(values, coords, n_rows, n_cols, patch_size_l0):
    """패치 좌표를 그리드 맵으로 변환"""
    spatial = np.full((n_rows, n_cols), np.nan)
    for i, (x, y) in enumerate(coords):
        gx = int(x / patch_size_l0)
        gy = int(y / patch_size_l0)
        if 0 <= gy < n_rows and 0 <= gx < n_cols:
            spatial[gy, gx] = values[i]
    return spatial

# 예측 Proportion spatial map
fig, axes = plt.subplots(3, 5, figsize=(30, 18))
fig.suptitle('Predicted Proportion — Spatial Heatmap', fontsize=18, y=1.01)

for gi, gene in enumerate(MARKER_GENES):
    ax = axes[gi // 5, gi % 5]
    spatial = make_spatial_map(all_pred_prop[:, gi], all_coords, n_rows, n_cols, patch_size_l0)
    im = ax.imshow(spatial, cmap='Blues', vmin=0, vmax=np.nanpercentile(spatial, 99),
                   interpolation='nearest', aspect='auto')
    ax.set_title(f'{gene}', fontsize=12)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
# ===== 12. Spatial Heatmap: 예측 Intensity =====
fig, axes = plt.subplots(3, 5, figsize=(30, 18))
fig.suptitle('Predicted Intensity — Spatial Heatmap', fontsize=18, y=1.01)

for gi, gene in enumerate(MARKER_GENES):
    ax = axes[gi // 5, gi % 5]
    spatial = make_spatial_map(all_pred_int[:, gi], all_coords, n_rows, n_cols, patch_size_l0)
    im = ax.imshow(spatial, cmap='Oranges', vmin=0, vmax=np.nanpercentile(spatial, 99),
                   interpolation='nearest', aspect='auto')
    ax.set_title(f'{gene}', fontsize=12)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
# ===== 13. GT vs Pred Spatial Heatmap 비교 (Proportion, available genes) =====
avail_genes = [(gi, gene) for gi, gene in available_markers]
n_avail = len(avail_genes)

fig, axes = plt.subplots(n_avail, 2, figsize=(20, 5 * n_avail))
fig.suptitle('Proportion: GT (left) vs Pred (right) — Spatial Heatmap', fontsize=18, y=1.005)

for row, (gi, gene) in enumerate(avail_genes):
    gt_map = make_spatial_map(gt_prop[:, gi], all_coords, n_rows, n_cols, patch_size_l0)
    pred_map = make_spatial_map(all_pred_prop[:, gi], all_coords, n_rows, n_cols, patch_size_l0)
    vmax = max(np.nanpercentile(gt_map, 99), np.nanpercentile(pred_map, 99))

    ax = axes[row, 0]
    im = ax.imshow(gt_map, cmap='Blues', vmin=0, vmax=vmax, interpolation='nearest', aspect='auto')
    ax.set_title(f'{gene} — GT', fontsize=11)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    ax = axes[row, 1]
    im = ax.imshow(pred_map, cmap='Blues', vmin=0, vmax=vmax, interpolation='nearest', aspect='auto')
    pcc_val = pcc_prop_list[gi]
    ax.set_title(f'{gene} — Pred (PCC={pcc_val:.3f})', fontsize=11)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
# ===== 14. GT vs Pred Spatial Heatmap 비교 (Intensity, available genes) =====
fig, axes = plt.subplots(n_avail, 2, figsize=(20, 5 * n_avail))
fig.suptitle('Intensity: GT (left) vs Pred (right) — Spatial Heatmap', fontsize=18, y=1.005)

for row, (gi, gene) in enumerate(avail_genes):
    gt_map = make_spatial_map(gt_int[:, gi], all_coords, n_rows, n_cols, patch_size_l0)
    pred_map = make_spatial_map(all_pred_int[:, gi], all_coords, n_rows, n_cols, patch_size_l0)
    vmax = max(np.nanpercentile(gt_map, 99), np.nanpercentile(pred_map, 99))

    ax = axes[row, 0]
    im = ax.imshow(gt_map, cmap='Oranges', vmin=0, vmax=vmax, interpolation='nearest', aspect='auto')
    ax.set_title(f'{gene} — GT', fontsize=11)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    ax = axes[row, 1]
    im = ax.imshow(pred_map, cmap='Oranges', vmin=0, vmax=vmax, interpolation='nearest', aspect='auto')
    pcc_val = pcc_int_list[gi]
    ax.set_title(f'{gene} — Pred (PCC={pcc_val:.3f})', fontsize=11)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
# ===== 15. PCC & MAE 막대 차트 (available genes) =====
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
avail_names = [g for _, g in avail_genes]
avail_idx = [gi for gi, _ in avail_genes]
x = np.arange(len(avail_genes))
w = 0.35

ax = axes[0]
ax.bar(x - w/2, [pcc_prop_list[gi] for gi in avail_idx], w, label='Proportion', color='steelblue')
ax.bar(x + w/2, [pcc_int_list[gi] for gi in avail_idx], w, label='Intensity', color='coral')
ax.set_xticks(x); ax.set_xticklabels(avail_names, rotation=45, ha='right')
ax.set_ylabel('PCC'); ax.set_title('Per-Gene Pearson Correlation (WSI)')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0, color='gray', linewidth=0.5)

ax = axes[1]
ax.bar(x - w/2, [mae_prop_list[gi] for gi in avail_idx], w, label='Proportion', color='steelblue')
ax.bar(x + w/2, [mae_int_list[gi] for gi in avail_idx], w, label='Intensity', color='coral')
ax.set_xticks(x); ax.set_xticklabels(avail_names, rotation=45, ha='right')
ax.set_ylabel('MAE'); ax.set_title('Per-Gene Mean Absolute Error (WSI)')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# ===== 16. Scatter Plot: Pred vs GT (유효 유전자) =====
fig, axes = plt.subplots(2, len(avail_genes), figsize=(5 * len(avail_genes), 10))
fig.suptitle('Pred vs GT Scatter (WSI patches)', fontsize=16, y=1.02)

for col, (gi, gene) in enumerate(avail_genes):
    # Proportion
    ax = axes[0, col]
    ax.scatter(gt_prop_valid[:, gi], pred_prop_valid[:, gi], alpha=0.1, s=4, color='steelblue')
    ax.plot([0, 1], [0, 1], 'r--', linewidth=1)
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel('GT'); ax.set_ylabel('Pred')
    pcc_val = pcc_prop_list[gi]
    ax.set_title(f'{gene}\nProp PCC={pcc_val:.3f}', fontsize=9)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)

    # Intensity
    ax = axes[1, col]
    ax.scatter(gt_int_valid[:, gi], pred_int_valid[:, gi], alpha=0.1, s=4, color='coral')
    ax.plot([0, 1], [0, 1], 'r--', linewidth=1)
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel('GT'); ax.set_ylabel('Pred')
    pcc_val = pcc_int_list[gi]
    ax.set_title(f'{gene}\nInt PCC={pcc_val:.3f}', fontsize=9)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ===== 17. Overlay: 예측 Proportion을 이미지 위에 오버레이 =====
overlay_genes = ['EPCAM', 'FAP', 'CD3E', 'MS4A1', 'CD68', 'PECAM1']
overlay_idx = [MARKER_GENES.index(g) for g in overlay_genes]

# 저해상도 썸네일
thumb_w_target = 2000
thumb_scale_ov = thumb_w_target / W
thumb_for_overlay = np.array(Image.fromarray(slide_img).resize(
    (thumb_w_target, int(H * thumb_scale_ov)), Image.LANCZOS))
thumb_h, thumb_w = thumb_for_overlay.shape[:2]
sx = thumb_w / W
sy = thumb_h / H

fig, axes = plt.subplots(2, 3, figsize=(24, 16))
fig.suptitle('Overlay: Predicted Proportion on Image', fontsize=18, y=1.01)

for idx, (gi, gene) in enumerate(zip(overlay_idx, overlay_genes)):
    ax = axes[idx // 3, idx % 3]
    ax.imshow(thumb_for_overlay)

    cx = (all_coords[:, 0] + patch_size_l0 / 2) * sx
    cy = (all_coords[:, 1] + patch_size_l0 / 2) * sy
    vals = all_pred_prop[:, gi]

    scatter = ax.scatter(cx, cy, c=vals, cmap='hot', s=3, alpha=0.6,
                        vmin=0, vmax=np.percentile(vals, 99))
    plt.colorbar(scatter, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(f'{gene} (Proportion)', fontsize=13)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ===== 18. Overlay: 예측 Intensity를 이미지 위에 오버레이 =====
fig, axes = plt.subplots(2, 3, figsize=(24, 16))
fig.suptitle('Overlay: Predicted Intensity on Image', fontsize=18, y=1.01)

for idx, (gi, gene) in enumerate(zip(overlay_idx, overlay_genes)):
    ax = axes[idx // 3, idx % 3]
    ax.imshow(thumb_for_overlay)

    cx = (all_coords[:, 0] + patch_size_l0 / 2) * sx
    cy = (all_coords[:, 1] + patch_size_l0 / 2) * sy
    vals = all_pred_int[:, gi]

    scatter = ax.scatter(cx, cy, c=vals, cmap='hot', s=3, alpha=0.6,
                        vmin=0, vmax=np.percentile(vals, 99))
    plt.colorbar(scatter, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(f'{gene} (Intensity)', fontsize=13)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ===== 19. 유전자 그룹별 성능 요약 =====
group_names = list(GENE_GROUPS.keys())
group_pcc_prop = []
group_pcc_int = []
group_mae_prop = []
group_mae_int = []

for gname, indices in GENE_GROUPS.items():
    # 유효 유전자만 사용
    valid_g = [i for i in indices if not np.isnan(pcc_prop_list[i])]
    if valid_g:
        group_pcc_prop.append(np.mean([pcc_prop_list[i] for i in valid_g]))
        group_pcc_int.append(np.mean([pcc_int_list[i] for i in valid_g]))
    else:
        group_pcc_prop.append(np.nan)
        group_pcc_int.append(np.nan)
    group_mae_prop.append(np.mean([mae_prop_list[i] for i in indices]))
    group_mae_int.append(np.mean([mae_int_list[i] for i in indices]))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
gx = np.arange(len(group_names))
gw = 0.35

ax = axes[0]
ax.bar(gx - gw/2, group_pcc_prop, gw, label='Proportion', color='steelblue')
ax.bar(gx + gw/2, group_pcc_int, gw, label='Intensity', color='coral')
ax.set_xticks(gx); ax.set_xticklabels(group_names, rotation=30, ha='right')
ax.set_ylabel('Mean PCC'); ax.set_title('PCC by Gene Group (WSI)')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0, color='gray', linewidth=0.5)

ax = axes[1]
ax.bar(gx - gw/2, group_mae_prop, gw, label='Proportion', color='steelblue')
ax.bar(gx + gw/2, group_mae_int, gw, label='Intensity', color='coral')
ax.set_xticks(gx); ax.set_xticklabels(group_names, rotation=30, ha='right')
ax.set_ylabel('Mean MAE'); ax.set_title('MAE by Gene Group (WSI)')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# ===== 20. Summary Table =====
print('=' * 80)
print('WSI-Level Inference Summary')
print('=' * 80)
print(f'Slide: {os.path.basename(slide_path)}')
print(f'Slide dimensions: {W} x {H} (MPP={slide_mpp:.4f})')
print(f'Total tissue patches: {len(all_coords)}')
print(f'Patches with Xenium cells: {valid_mask.sum()}')
print(f'Available marker genes: {len(available_markers)}/{NUM_GENES}')
print()

print(f'{"Metric":<25s} | {"Proportion":>12s} | {"Intensity":>12s}')
print('-' * 55)
valid_pcc_p = [p for p in pcc_prop_list if not np.isnan(p)]
valid_pcc_i = [p for p in pcc_int_list if not np.isnan(p)]
print(f'{"Per-Gene PCC (mean)":<25s} | {np.mean(valid_pcc_p):>12.4f} | {np.mean(valid_pcc_i):>12.4f}')
print(f'{"Per-Gene MAE (mean)":<25s} | {np.mean(mae_prop_list):>12.4f} | {np.mean(mae_int_list):>12.4f}')

# Per-sample PCC
sample_pcc_prop = []
sample_pcc_int = []
for si in range(len(pred_prop_valid)):
    try:
        rp, _ = stats.pearsonr(pred_prop_valid[si], gt_prop_valid[si])
        ri, _ = stats.pearsonr(pred_int_valid[si], gt_int_valid[si])
        sample_pcc_prop.append(rp)
        sample_pcc_int.append(ri)
    except:
        pass
sample_pcc_prop = np.array(sample_pcc_prop)
sample_pcc_int = np.array(sample_pcc_int)
print(f'{"Per-Sample PCC (mean)":<25s} | {sample_pcc_prop.mean():>12.4f} | {sample_pcc_int.mean():>12.4f}')
print(f'{"Per-Sample PCC (std)":<25s} | {sample_pcc_prop.std():>12.4f} | {sample_pcc_int.std():>12.4f}')
print('=' * 80)

: 